In [109]:
import pandas as pd
import numpy as np

In [110]:
from google.colab import files

uploaded = files.upload()

Saving tiny-shakespeare.txt to tiny-shakespeare (4).txt


In [111]:
with open ("tiny-shakespeare.txt","r",encoding="UTF-8") as f:
    text=f.read()

In [112]:
len(text)

1115394

In [113]:
text[:1000:]

"First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us kill him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be done: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor citizens, the patricians good.\nWhat authority surfeits on would relieve us: if they\nwould yield us but the superfluity, while it were\nwholesome, we might guess they relieved us humanely;\nbut they think we are too dear: the leanness that\nafflicts us, the object of our misery, is as an\ninventory to particularise their abundance; our\nsufferance is a gain to them Let us revenge this with\nour pikes, ere we become rakes: for the gods know I\nspeak this in hunger 

In [114]:
import re
text=text.lower()
text=re.sub(r"\n+"," ",text)

In [115]:
len(text)

1108171

In [116]:
text[:1000:]

"first citizen: before we proceed any further, hear me speak. all: speak, speak. first citizen: you are all resolved rather to die than to famish? all: resolved. resolved. first citizen: first, you know caius marcius is chief enemy to the people. all: we know't, we know't. first citizen: let us kill him, and we'll have corn at our own price. is't a verdict? all: no more talking on't; let it be done: away, away! second citizen: one word, good citizens. first citizen: we are accounted poor citizens, the patricians good. what authority surfeits on would relieve us: if they would yield us but the superfluity, while it were wholesome, we might guess they relieved us humanely; but they think we are too dear: the leanness that afflicts us, the object of our misery, is as an inventory to particularise their abundance; our sufferance is a gain to them let us revenge this with our pikes, ere we become rakes: for the gods know i speak this in hunger for bread, not in thirst for revenge. second ci

In [117]:
tokens=text.split()

In [118]:
from collections import Counter
word_counts = Counter(tokens)
print(word_counts.most_common(10))

[('the', 6279), ('and', 5479), ('to', 4723), ('i', 4403), ('of', 3721), ('my', 3114), ('a', 2975), ('you', 2449), ('that', 2427), ('in', 2312)]


In [119]:
vocab = ["<UNK>"] + sorted([w for w, c in word_counts.items() if c >= 2])
vocab_len = len(vocab)

In [120]:
vocab_len=len(vocab)

In [121]:
print(vocab_len)

10113


In [122]:
word_to_index = {}
for idx, word in enumerate(vocab):
    word_to_index[word] = idx

In [123]:
index_to_word = {idx: word for idx, word in enumerate(vocab)}


In [124]:
encoded_text = []
for word in tokens:
    encoded_text.append(word_to_index.get(word, word_to_index["<UNK>"]))

In [125]:
seq_length = 10

sequences = []
targets = []

for i in range(len(encoded_text) - seq_length):
    sequences.append(encoded_text[i : i + seq_length])
    targets.append(encoded_text[i + seq_length])

In [126]:
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader

X_tensor = torch.tensor(sequences, dtype=torch.long)
y_tensor = torch.tensor(targets, dtype=torch.long)

dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

In [127]:
class LSTM(nn.Module):
    def __init__(self,vocab_len,input_size,hidden_size,num_layers,dropout=0.4):
        super().__init__()

        self.input_size=input_size
        self.hidden_size=hidden_size
        self.num_layers=num_layers


        self.embedding=nn.Embedding(vocab_len,self.input_size)
        self.lstm=nn.LSTM(
            self.input_size,
            self.hidden_size,
            self.num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc=nn.Linear(self.hidden_size,vocab_len)

    def forward(self, x, hidden=None):
        x = self.embedding(x)
        out, hidden = self.lstm(x, hidden)
        out = out[:, -1, :]
        out = self.fc(out)
        return out, hidden

In [128]:
import torch

# Check if the T4 GPU is available, otherwise default to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}") # Make sure this outputs 'cuda'

# Instantiate the model
model = LSTM(vocab_len, input_size=128, hidden_size=256, num_layers=2,dropout=0.4)

# PUSH THE MODEL TO THE GPU
model = model.to(device)

import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),weight_decay=1e-4)

Using device: cuda


In [129]:
train_size=int(0.8*len(dataset))
test_size=len(dataset)-train_size

train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=True

)

In [130]:
best_val_loss=float('inf')

epochs=25

for epoch in range(epochs):
    model.train()
    epoch_loss=0.0

    for xb,yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()

        output, _=model(xb)
        loss=criterion(output,yb)
        loss.backward()
        optimizer.step()
        epoch_loss+=loss.item()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
      for xb, yb in test_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        output, _ = model(xb)
        loss = criterion(output, yb)
        val_loss += loss.item()

    avg_train_loss = epoch_loss / len(train_loader)
    avg_val_loss = val_loss / len(test_loader)

    print(f"epoch = {epoch+1} & train_loss = {epoch_loss/len(train_loader)} & val_loss = {val_loss/len(test_loader)}")

    if avg_val_loss<best_val_loss:
      best_val_loss=avg_val_loss
      torch.save(model.state_dict(),"shakespeare_lstm2.pth")
      print(f"new best model saved(val_loss={avg_val_loss})")



epoch = 1 & train_loss = 6.767290666770031 & val_loss = 6.533856364745128
new best model saved(val_loss=6.533856364745128)
epoch = 2 & train_loss = 6.371850904130258 & val_loss = 6.279082591020608
new best model saved(val_loss=6.279082591020608)
epoch = 3 & train_loss = 6.146598952264771 & val_loss = 6.116287880306002
new best model saved(val_loss=6.116287880306002)
epoch = 4 & train_loss = 5.971564637917854 & val_loss = 5.998530988451801
new best model saved(val_loss=5.998530988451801)
epoch = 5 & train_loss = 5.834002518917524 & val_loss = 5.916635474072227
new best model saved(val_loss=5.916635474072227)
epoch = 6 & train_loss = 5.732296853268881 & val_loss = 5.878782933271384
new best model saved(val_loss=5.878782933271384)
epoch = 7 & train_loss = 5.658613472376578 & val_loss = 5.848858039590377
new best model saved(val_loss=5.848858039590377)
epoch = 8 & train_loss = 5.602350706549429 & val_loss = 5.8329089744181575
new best model saved(val_loss=5.8329089744181575)
epoch = 9 & tr

In [131]:
import math

model.load_state_dict(torch.load("shakespeare_lstm2.pth"))
model.eval()
with torch.no_grad():
    corr_val = 0
    total_val = 0
    total_loss = 0.0

    for xb, yb in test_loader:

        xb = xb.to(device)
        yb = yb.to(device)
        output, _ = model(xb)
        loss = criterion(output, yb)

        preds = output.argmax(dim=1)
        total_val += yb.size(0)
        corr_val += (preds == yb).sum().item()
        total_loss += loss.item()

avg_loss = total_loss / len(test_loader)
perplexity = math.exp(avg_loss)

print(f"accuracy = {corr_val/total_val*100}%")
print(f"perplexity = {perplexity}")

accuracy = 12.794204905063292%
perplexity = 320.11273860643746


In [79]:
import torch
print(torch.cuda.is_available())

True


In [80]:
import torch

# Save only the model weights (the most efficient way)
torch.save(model.state_dict(), 'shakespeare_lstm2.pth')
print("Model successfully saved to shakespeare_lstm2.pth!")

Model successfully saved to shakespeare_lstm2.pth!


In [132]:
from google.colab import files

# This will trigger a browser download for the file
files.download('shakespeare_lstm2.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>